# 02 — Retrieval ablations (Stages 1-7)

Each stage varies **one** thing and holds everything else at the current best-known
configuration, which is threaded forward as each stage picks a winner. All of it runs
through the same `run_config` entry point, so the numbers are comparable across stages
rather than each stage having its own bespoke measurement path.

The tables below are the artefacts written by `scripts/run_retrieval_ablations.py`.

In [ ]:
import sys, json; sys.path.insert(0, "../src")
import pandas as pd
from pathlib import Path

RESULTS = Path("../reports/results")
def table(stage):
    return pd.DataFrame(json.loads((RESULTS / f"{stage}.json").read_text()))

table("stage1_parsing")

**Stage 1.** All three parsers extract all 45 pages and clear the clean-text bar, so
the usual "clean text %" column does not separate them. The column that does is
gold-span recoverability: pypdf loses one of the 30 answer spans, rendering `those` as
`t hose`. PyMuPDF wins on 100% recovery *and* is ~15× faster than pdfplumber.

In [ ]:
table("stage2_chunking").sort_values(["R@3", "MRR@10"], ascending=False).head(8)

**Stage 2 is a near-tie.** The top four configurations share R@3 = 0.9000 and are
separated by <0.01 MRR@10 — under a third of one question out of 30.

`rule_aware` was a custom strategy built to split on the documents' own `RULE 20:` /
`G14` headings, on the theory that a legal rule boundary is a real semantic boundary.
It did **not** win. Both tariffs are already written as short numbered rules, so any
sane splitter lands boundaries in roughly the right places. A useful negative result
about how much document-specific engineering this corpus actually rewards.

In [ ]:
display(table("stage3_embeddings"))
display(table("stage4_vectordb"))

**Stage 3** is the largest clean win in the canonical stages: bge-small beats MiniLM by
+0.1333 R@3 for 0.4 ms more per query. Both bge and nomic are *asymmetric* models
requiring different query and document prefixes; those are applied, because
benchmarking them without their prefixes is a common error that would have understated
both and handed the stage to MiniLM by default.

**Stage 4** behaves exactly as the methodology predicts — identical embeddings retrieve
identically, so R@3 is a parity check and the decision is made on operational grounds.

## Stage 5 — the most important table in the study

In [ ]:
t5 = table("stage5_retrieval_mode")
t5[["Mode", "R@3 (all)", "R@3 (keyword)", "R@3 (semantic)",
    "R@3 (passenger)", "R@3 (cargo)", "R@3 (financial)", "MRR@10", "NDCG@3"]]

This is the evidence that justifies hybrid, and it is the per-route breakdown this
group's `REQUIREMENT.md` specifically asks for.

The two retrievers have **opposite** weaknesses:

- **Sparse** is perfect on passenger (1.0000) — those questions quote tariff
  vocabulary almost verbatim, which is BM25's home ground — and collapses on
  financial (0.7143).
- **Dense** is the reverse: strong on financial (0.8571), weakest on cargo (0.5455).

Hybrid does not split the difference. On semantic queries it **beats both**
(0.9375 vs 0.7500 dense and 0.8750 sparse). Had one mode dominated every column,
running two retrievers and fusing them would have been unjustifiable complexity.

In [ ]:
display(table("stage6_fusion"))
display(table("stage7_reranking"))

**Stage 6 — a win worth exactly one question.** Weighted α = 0.5 leads RRF on R@3 by
0.0333, which on 30 questions *is* one question. Note also that the α sweep is
**non-monotonic**: that is what overfitting to a small set looks like, not a tuned
optimum. RRF has no hyperparameter and is invariant to score scale, so it remains the
safer production choice; the measured winner is reported as measured with the caveat
attached.

**Stage 7 — the reranker is rejected.** It *lowers* NDCG@3 and costs 120 ms to do it,
and the damage grows with the candidate pool. Two reasons:

1. **Domain mismatch.** `ms-marco-MiniLM` is trained on short factual web-search
   passages. This corpus is dense legal tariff prose full of cross-references and
   XBRL label-plus-numbers records. Its learned notion of relevance does not transfer,
   so its confident reordering is confidently wrong.
2. **No headroom.** Fusion already reaches R@3 = 0.90. A reranker cannot add documents
   fusion never retrieved — it can only reshuffle. When the right chunk is usually
   already in the top 3, reshuffling is more likely to demote it than promote it.

Adding a reranker because a tutorial did would have cost this pipeline both accuracy
and latency.

## The two changes that mattered more than any of the eight stages

Both were found by investigating a *failure*, not by sweeping hyperparameters.

In [ ]:
display(table("stage0_xbrl"))
display(table("stage0b_boilerplate"))

**Stage 0 — and a hypothesis that was wrong.** The financial route scored R@3 = 0.1429
on sparse retrieval. Dense and sparse fail in *different* ways, so their failing
identically pointed at the corpus rather than the config.

The obvious culprit was boilerplate: 420 records sharing a header and repeating
`for the period 2025-01-01 to 2025-12-31` on every line. Variant B removes all of it
and changed **nothing** (0.1429 → 0.1429). That hypothesis was wrong, and the table
records it as wrong.

The real defect was a **vocabulary gap**: US-GAAP names the element `Revenue from
Contract with Customer, Excluding Assessed Tax`, a human asks for `total operating
revenue`, and those strings share almost no terms — fatal for BM25, merely bad for
embeddings. Bridging it took sparse 0.1429 → 0.7143.

**Stage 0b — page furniture caused the routing failure.** A contents page lists every
rule title in the document, so it matches almost any topical query with high lexical
density while containing no answer — a chunk engineered to win retrieval and then say
nothing. And `Delta Domestic General Rules Tariff` appears on all 23 pages of the
passenger contract, indexing "domestic", "rules" and "tariff" 23 times, which is why a
*cargo* question phrased "rules for shipping cargo domestically" was pulled to the
passenger document. Dropping 2 of 45 pages fixed a failing required acceptance test.